In [ ]:
import os

import pandas as pd
import requests
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import plotly.graph_objects as go
from plotly.subplots import make_subplots


DARK_BG = "#0f172a"
PANEL_BG = "rgba(15, 23, 42, 0.96)"
TEXT = "white"
THEME = {
    "Temperature": "#ff6b6b",
    "Humidity": "#4dabf7",
    "Atmospheric Pressure": "#51cf66",
}


def fetch_openweathermap_forecast(city, api_key, units="metric"):
    """Fetch 5-day forecast data from OpenWeatherMap for a city."""
    if not api_key:
        raise ValueError("OpenWeatherMap API key is required.")

    url = "https://api.openweathermap.org/data/2.5/forecast"
    params = {
        "q": city,
        "appid": api_key,
        "units": units,
    }
    response = requests.get(url, params=params, timeout=20)

    if not response.ok:
        try:
            message = response.json().get("message", response.text)
        except Exception:
            message = response.text
        raise RuntimeError(f"OpenWeatherMap request failed ({response.status_code}): {message}")

    payload = response.json()

    rows = []
    for entry in payload["list"]:
        rows.append(
            {
                "time": pd.to_datetime(entry["dt_txt"]),
                "temperature": entry["main"]["temp"],
                "humidity": entry["main"]["humidity"],
                "pressure": entry["main"]["pressure"],
                "description": entry["weather"][0]["description"].title(),
            }
        )

    weather_frame = pd.DataFrame(rows)
    city_info = payload.get("city", {})
    coord = city_info.get("coord", {})
    return weather_frame, {
        "city": city_info.get("name", city),
        "country": city_info.get("country", ""),
        "lat": coord.get("lat"),
        "lon": coord.get("lon"),
    }


def build_insight_paragraph(weather_frame, city):
    """Create a short insight paragraph summarizing the weather chart."""
    latest = weather_frame.iloc[-1]
    first = weather_frame.iloc[0]
    temp_delta = latest["temperature"] - first["temperature"]
    humidity_delta = latest["humidity"] - first["humidity"]
    pressure_delta = latest["pressure"] - first["pressure"]

    temp_trend = "rising" if temp_delta > 0 else "falling" if temp_delta < 0 else "stable"
    humidity_trend = "increasing" if humidity_delta > 0 else "decreasing" if humidity_delta < 0 else "steady"
    pressure_trend = "climbing" if pressure_delta > 0 else "easing" if pressure_delta < 0 else "unchanged"

    return (
        f"<div style='background:{PANEL_BG}; border:1px solid rgba(148,163,184,0.35); border-radius:16px; padding:14px 16px; color:{TEXT}; "
        f"box-shadow:0 14px 34px rgba(15,23,42,0.18); margin-bottom:14px;'>"
        f"<div style='font-size:16px; font-weight:700; margin-bottom:6px;'>Quick Insight</div>"
        f"<div style='font-size:13px; line-height:1.55; opacity:0.95;'>"
        f"For {city}, the latest reading shows {latest['description'].lower()} conditions with temperature {temp_trend} overall, "
        f"humidity {humidity_trend}, and pressure {pressure_trend}. The values are best read together: temperature shows the main short-term movement, "
        f"humidity explains how moist the air is, and pressure gives a useful clue about changing weather patterns."
        f"</div></div>"
    )


def build_cards(weather_frame):
    max_temp = weather_frame["temperature"].max()
    min_temp = weather_frame["temperature"].min()
    avg_temp = weather_frame["temperature"].mean()
    temp_change = weather_frame["temperature"].iloc[-1] - weather_frame["temperature"].iloc[0]
    change_text = "increase" if temp_change > 0 else "decrease" if temp_change < 0 else "no change"
    change_value = abs(temp_change)

    def card(title, value, subtitle, accent):
        return f"""
        <div style="
            flex: 1;
            min-width: 180px;
            background: linear-gradient(180deg, rgba(15,23,42,0.98), rgba(30,41,59,0.95));
            border: 1px solid rgba(148,163,184,0.22);
            border-top: 4px solid {accent};
            border-radius: 18px;
            padding: 18px 16px;
            color: white;
            box-shadow: 0 14px 30px rgba(15,23,42,0.20);
        ">
            <div style="font-size: 13px; opacity: 0.82; margin-bottom: 8px;">{title}</div>
            <div style="font-size: 30px; font-weight: 800; line-height: 1; margin-bottom: 8px;">{value}</div>
            <div style="font-size: 12px; opacity: 0.88;">{subtitle}</div>
        </div>
        """

    return HTML(
        f"""
        <div style="display: flex; gap: 12px; flex-wrap: wrap; margin-bottom: 14px;">
            {card('Highest Temperature', f'{max_temp:.1f}°', 'Peak from the live forecast window', '#ff6b6b')}
            {card('Lowest Temperature', f'{min_temp:.1f}°', 'Coolest reading in the forecast window', '#4dabf7')}
            {card('Average Temperature', f'{avg_temp:.1f}°', 'Mean forecast temperature', '#51cf66')}
            {card('Temp Change', f'{change_value:.1f}° {change_text}', 'Start to end movement in the forecast period', '#f59e0b')}
        </div>
        """
    )


def plot_weather_series(weather_frame, visual_type="Line", city="", city_meta=None):
    """Plot weather data as three clear charts with hover tooltips plus map and summary visual."""
    if weather_frame.empty:
        raise ValueError("No weather data available to plot.")

    x_values = weather_frame["time"].dt.strftime("%b %d, %Y %H:%M")

    fig = make_subplots(
        rows=3,
        cols=2,
        column_widths=[0.68, 0.32],
        row_heights=[0.33, 0.33, 0.34],
        shared_xaxes=True,
        vertical_spacing=0.07,
        horizontal_spacing=0.10,
        specs=[
            [{}, {"type": "mapbox"}],
            [{}, {"type": "domain"}],
            [{}, {"type": "table"}],
        ],
        subplot_titles=("Temperature", "City Location", "Humidity", "Pressure Trend", "Atmospheric Pressure", "Forecast Snapshot"),
    )

    chart_map = [
        (1, "Temperature", weather_frame["temperature"]),
        (2, "Humidity", weather_frame["humidity"]),
        (3, "Atmospheric Pressure", weather_frame["pressure"]),
    ]

    for row, label, values in chart_map:
        if visual_type == "Bar":
            trace = go.Bar(
                x=x_values,
                y=values,
                name=label,
                marker_color=THEME[label],
                hovertemplate=f"<b>{label}</b><br>Time: %{{x}}<br>Value: %{{y}}<extra></extra>",
                showlegend=False,
            )
        elif visual_type == "Area":
            trace = go.Scatter(
                x=x_values,
                y=values,
                mode="lines",
                name=label,
                line=dict(color=THEME[label], width=3),
                fill="tozeroy",
                hovertemplate=f"<b>{label}</b><br>Time: %{{x}}<br>Value: %{{y}}<extra></extra>",
                showlegend=False,
            )
        elif visual_type == "Scatter":
            trace = go.Scatter(
                x=x_values,
                y=values,
                mode="markers+lines",
                name=label,
                line=dict(color=THEME[label], width=2),
                marker=dict(size=10, color=THEME[label]),
                hovertemplate=f"<b>{label}</b><br>Time: %{{x}}<br>Value: %{{y}}<extra></extra>",
                showlegend=False,
            )
        else:
            trace = go.Scatter(
                x=x_values,
                y=values,
                mode="lines+markers",
                name=label,
                line=dict(color=THEME[label], width=3),
                marker=dict(size=8),
                hovertemplate=f"<b>{label}</b><br>Time: %{{x}}<br>Value: %{{y}}<extra></extra>",
                showlegend=False,
            )

        fig.add_trace(trace, row=row, col=1)

    city_meta = city_meta or {}
    lat = city_meta.get("lat")
    lon = city_meta.get("lon")
    label = city_meta.get("city", city)
    country = city_meta.get("country", "")
    if lat is not None and lon is not None:
        fig.add_trace(
            go.Scattermapbox(
                lat=[lat],
                lon=[lon],
                mode="markers",
                marker=dict(size=18, color="#f97316"),
                text=[f"{label} {country}".strip()],
                hovertemplate=f"<b>{label} {country}</b><br>Lat: {lat}<br>Lon: {lon}<extra></extra>",
                showlegend=False,
            ),
            row=1,
            col=2,
        )
        fig.update_mapboxes(
            style="carto-positron",
            zoom=4,
            center=dict(lat=lat, lon=lon),
        )
    else:
        fig.add_annotation(
            text="Location unavailable from API response",
            xref="x domain",
            yref="y domain",
            x=0.5,
            y=0.5,
            showarrow=False,
            row=1,
            col=2,
        )

    pressure_change = weather_frame["pressure"].iloc[-1] - weather_frame["pressure"].iloc[0]
    pressure_direction = "up" if pressure_change > 0 else "down" if pressure_change < 0 else "flat"
    fig.add_trace(
        go.Indicator(
            mode="number+delta",
            value=float(weather_frame["pressure"].iloc[-1]),
            delta={"reference": float(weather_frame["pressure"].iloc[0]), "increasing": {"color": "#51cf66"}, "decreasing": {"color": "#ff6b6b"}},
            number={"suffix": " hPa", "font": {"size": 34, "color": "white"}},
            title={"text": f"Pressure moved {pressure_direction}", "font": {"size": 14, "color": "white"}},
        ),
        row=2,
        col=2,
    )

    latest_rows = weather_frame[["time", "temperature", "humidity", "pressure", "description"]].tail(5)
    fig.add_trace(
        go.Table(
            header=dict(values=["Time", "Temp", "Hum", "Press", "Condition"], fill_color="#1d4ed8", font=dict(color="white"), align="center"),
            cells=dict(
                values=[
                    latest_rows["time"].dt.strftime("%b %d %H:%M"),
                    latest_rows["temperature"].map(lambda v: f"{v:.1f}"),
                    latest_rows["humidity"].astype(str),
                    latest_rows["pressure"].astype(str),
                    latest_rows["description"],
                ],
                fill_color="#1e293b",
                font=dict(color="white"),
                align="center",
            ),
        ),
        row=3,
        col=2,
    )

    fig.update_layout(
        template="plotly_dark",
        paper_bgcolor=DARK_BG,
        plot_bgcolor="#111827",
        title=dict(text=f"Dynamic Weather Dashboard for {city or 'Selected City'}", x=0.5),
        font=dict(color=TEXT),
        height=980,
        margin=dict(l=40, r=40, t=90, b=40),
        hovermode="x unified",
    )
    fig.update_xaxes(showgrid=True, gridcolor="rgba(255,255,255,0.08)", row=3, col=1)
    fig.update_yaxes(showgrid=True, gridcolor="rgba(255,255,255,0.08)")
    fig.show()


def build_weather_dashboard():
    header = HTML(
        """
        <div style="
            background: linear-gradient(135deg, #0f172a, #1d4ed8 55%, #06b6d4);
            color: white;
            padding: 22px 26px;
            border-radius: 20px;
            box-shadow: 0 18px 40px rgba(15, 23, 42, 0.35);
            margin-bottom: 14px;
            border: 1px solid rgba(255,255,255,0.12);
        ">
            <div style="font-size: 30px; font-weight: 800; letter-spacing: 0.5px;">Weather Vision Studio</div>
            <div style="font-size: 14px; opacity: 0.9; margin-top: 6px;">Group 5 Weather dashboard | live weather insights</div>
        </div>
        """
    )

    city_input = widgets.Text(value="London", description="City", layout=widgets.Layout(width="240px"))
    api_key_input = widgets.Password(
        value=os.getenv("OPENWEATHERMAP_API_KEY", ""),
        description="API Key",
        layout=widgets.Layout(width="320px"),
    )
    visual_selector = widgets.ToggleButtons(
        options=["Line", "Bar", "Area", "Scatter"],
        value="Line",
        description="Visual",
        button_style="info",
    )
    load_button = widgets.Button(
        description="Load Weather Data",
        button_style="success",
        icon="cloud",
        layout=widgets.Layout(width="180px"),
    )
    output = widgets.Output()
    insight_output = widgets.HTML()
    cards_output = widgets.Output()

    panel_style = {
        "border": "1px solid rgba(148, 163, 184, 0.35)",
        "border_radius": "18px",
        "padding": "14px",
        "background": PANEL_BG,
        "box_shadow": "0 14px 34px rgba(15, 23, 42, 0.18)",
    }
    controls = widgets.VBox(
        [
            widgets.HBox([city_input, api_key_input]),
            widgets.HBox([visual_selector, load_button]),
        ],
        layout=widgets.Layout(**panel_style),
    )
    output_box = widgets.Box([output], layout=widgets.Layout(**panel_style, min_height="420px"))

    state = {"weather_frame": None, "city_meta": None}

    def draw_weather(_=None):
        with output:
            clear_output(wait=True)
            try:
                weather_frame, city_meta = fetch_openweathermap_forecast(
                    city_input.value.strip(),
                    api_key_input.value.strip(),
                )
                state["weather_frame"] = weather_frame
                state["city_meta"] = city_meta
                insight_output.value = build_insight_paragraph(weather_frame, city_input.value.strip())
                with cards_output:
                    clear_output(wait=True)
                    display(build_cards(weather_frame))
                plot_weather_series(weather_frame, visual_selector.value, city_input.value.strip(), city_meta)
            except Exception as exc:
                insight_output.value = ""
                with cards_output:
                    clear_output(wait=True)
                print("Unable to load live OpenWeatherMap data.")
                print(f"Error: {exc}")
                print("Check that the API key is active, the city name is valid, and your OpenWeatherMap account can access the forecast endpoint.")

    def on_load_clicked(_):
        draw_weather()

    def on_visual_change(change):
        if state["weather_frame"] is not None:
            with output:
                clear_output(wait=True)
                plot_weather_series(state["weather_frame"], visual_selector.value, city_input.value.strip(), state["city_meta"])

    load_button.on_click(on_load_clicked)
    visual_selector.observe(on_visual_change, names="value")

    display(header, insight_output, cards_output, controls, output_box)
    draw_weather()


build_weather_dashboard()

HTML(value='')

Output()

Box(children=(Output(),), layout=Layout(border_bottom='1px solid rgba(148, 163, 184, 0.35)', border_left='1px …